# 13주차 실습 — 의료 질 · 간호 지표 만들기

- **데이터**: 가상 병원 CDW (2024년 · 병동 6곳). 실제 병원 자료가 아닌 수업용 가상 자료입니다.
- **오늘 배우는 것**: 지표 정의서, 30일 재입원율, 근무조별 간호사 1인당 환자 수, 작은 칸 가리기

## 0. 시작하기 전에

- Colab 에서는 먼저 **`File → Save a copy in Drive`** 를 누르세요.
- 셀은 **위에서부터 차례로** 실행합니다 (▶ 또는 `Shift + Enter`). `%%sql` 아래의 `-- Q1` · `-- S1` 같은 문항 이름 줄은 지우지 마세요.
- 문항 SQL 을 실행하면 결과 아래에 **✅ / ❌** 가 바로 나옵니다.
- 오류가 나면 **`… Error:` 로 시작하는 줄**만 읽으면 됩니다.

In [ ]:
#@title [1번] ▶ 준비 셀 — 실행만 하세요 (Shift+Enter). 코드는 몰라도 됩니다. { display-mode: "form" }
import os, sys, urllib.request
import numpy as _np, pandas as _pd
DB  = "hospital_mini.duckdb"
URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/data/hospital_mini.duckdb.gz"

# 로컬(수업 폴더)에서 열었을 때는 ../data/ 의 파일을 사용
for cand in [DB, os.path.join("..", "data", DB), os.path.join("data", DB)]:
    if os.path.exists(cand):
        DB = cand
        break
else:
    print("데이터 파일을 내려받습니다 (약 1MB) ...")
    import gzip, shutil
    urllib.request.urlretrieve(URL, DB + ".gz")
    with gzip.open(DB + ".gz", "rb") as f_in, open(DB, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    os.remove(DB + ".gz")

if "google.colab" in sys.modules:           # Colab 에서만 설치 (수업에서 검증한 버전)
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb==1.4.5", "jupysql==0.11.1", "duckdb-engine==0.17.0"])

import duckdb
_pd.set_option("display.max_colwidth", 60)
_pd.set_option("display.max_columns", None)     # SELECT * 로 열 이름을 볼 때 모든 열을 보여 줌
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 30
%config SqlMagic.feedback = 0
%xmode Minimal
try:                                         # jupysql 의 영어 안내문(If using snippets ...)을 숨김
    import sql.error_handler as _eh
    _eh.CTE_MSG, _eh.ORIGINAL_ERROR = "↓ 아래 '… Error:' 로 시작하는 줄이 원인입니다", ""
except Exception:
    pass
conn = duckdb.connect(DB, read_only=True)
conn.execute("SET python_enable_replacements = false")   # 빈칸 ___ 을 파이썬 변수로 읽지 않게

class _SameCursor:                           # jupysql 이 셀마다 새 cursor(=새 연결)를 열면
    def __init__(self, c): self._c = c       # CREATE TEMP TABLE 로 만든 표가 다음 셀에서 사라지므로
    def close(self): pass                    # 항상 같은 연결을 쓰게 합니다
    def __getattr__(self, k): return getattr(self._c, k)

class _OneConnection:
    def __init__(self, c): self._c = c
    def cursor(self): return _SameCursor(self._c)
    def __getattr__(self, k): return getattr(self._c, k)

_aact = _OneConnection(conn)
%sql _aact --alias aact

import hashlib as _hl, math as _math, decimal as _dec, datetime as _dt
import re as _re, zlib as _zlib, base64 as _b64, json as _json

def _load_answers(blob):
    """자동 확인용 지문(정답 결과표의 해시 · 행 수 · 열 이름 · 힌트)을 풉니다. 정답 SQL 은 들어 있지 않습니다."""
    return _json.loads(_zlib.decompress(_b64.b64decode(blob)).decode())

def _v(v):
    """값 하나를 비교용 문자열로: 빈 값 → '', 1947.0 → '1947', 날짜 → 'YYYY-MM-DD'"""
    try:
        if v is None or (not isinstance(v, (str, bytes)) and _pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, _dec.Decimal):
        v = float(v)
    if isinstance(v, (bool, _np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, _np.integer)):
        return str(int(v))
    if isinstance(v, (float, _np.floating)):
        v = round(float(v), 4)
        return str(int(v)) if v.is_integer() else repr(v)
    if isinstance(v, (_pd.Timestamp, _dt.datetime)):
        if v.hour == 0 and v.minute == 0 and v.second == 0 and v.microsecond == 0:
            return v.strftime("%Y-%m-%d")
        return v.isoformat()
    if isinstance(v, _dt.date):
        return v.isoformat()
    return str(v)

def _sigs(df):
    """결과표의 지문: 열 이름(소문자), 행 수, 행 순서 무시 지문(sig), 행 순서 포함 지문(osig)"""
    cols = [str(c).strip().lower() for c in df.columns]
    rows = ["\x1f".join(_v(x) for x in r) for r in df.itertuples(index=False, name=None)]
    head = "\x1e".join(cols)
    h = lambda rs: _hl.md5((head + "\x1d" + "\x1e".join(rs)).encode()).hexdigest()[:12]
    return {"sig": h(sorted(rows)), "osig": h(rows), "rows": len(rows), "cols": cols}

_LAST = {"tag": None, "error": False, "sql": ""}
_TRIES, _RESULT = {}, {}

def _after_cell(result):
    info = getattr(result, "info", None)
    src = (getattr(info, "raw_cell", "") or "").lstrip()
    if not src.startswith("%%sql"):
        return
    tag = None
    for line in src.split("\n")[1:]:
        s = line.strip()
        if s.startswith("--"):
            parts = s[2:].split()
            if parts and _re.match(r"^\[\d+번\]$", parts[0]):      # 수업용 셀 번호 [12번] 은 건너뜀
                parts = parts[1:]
            tag = parts[0] if parts and not parts[0].startswith("[") else None
            break
        if s:
            break
    body = "\n".join(l.split("--")[0] for l in src.split("\n")[1:])
    _LAST.update(tag=tag, error=not result.success, sql=body)
    if tag and tag in globals().get("ANSWERS", {}):         # 문항 셀(-- Q1)이면 실행하자마자 ✅/❌ 표시
        check(tag, getattr(result, "result", None))

for _cb in list(get_ipython().events.callbacks["post_run_cell"]):
    if getattr(_cb, "__name__", "") == "_after_cell":
        get_ipython().events.unregister("post_run_cell", _cb)
get_ipython().events.register("post_run_cell", _after_cell)

def _expected(name):
    a = ANSWERS[name]
    if "by_mesh" in a:
        if not globals().get("MY_MESH"):
            return None
        return dict(a["by_mesh"][MY_MESH], hint=a.get("hint", ""), ordered=a.get("ordered", False))
    return a

def _miss(name, why, exp):
    """틀렸을 때: 기록하고, 두 번째부터 힌트를 붙여 출력"""
    _RESULT[name] = False
    _TRIES[name] = _TRIES.get(name, 0) + 1
    msg = f"❌ {name}  {why}"
    hint = (exp or ANSWERS.get(name, {})).get("hint")
    if hint:
        msg += ("\n   💡 힌트: " + hint) if _TRIES[name] >= 2 else "\n   (한 번 더 다르면 힌트가 나옵니다)"
    print(msg)

def check(name, df=None):
    """바로 위 %%sql 셀의 결과를 확인합니다.  예) check("Q1", _)"""
    if name not in ANSWERS:
        print(f"알 수 없는 문항 이름입니다: {name}"); return
    if _LAST["tag"] and _LAST["tag"] != name:
        print(f"⚠️ 방금 실행한 SQL 셀은 {_LAST['tag']} 입니다. {name} 의 %%sql 셀을 먼저 실행한 뒤 이 셀을 실행하세요."); return
    exp = _expected(name)
    if exp is None:
        print("⚠️ 분석 질환 정보가 없습니다. 맨 위 준비 셀을 다시 실행하세요."); return
    if _LAST["sql"] and not _re.search(r"\bFROM\b", _LAST["sql"], _re.I):
        return _miss(name, "FROM 이 없는 SQL 입니다. 숫자를 직접 적지 말고, 표에서 꺼내 세는 SQL 을 쓰세요.", exp)
    if _LAST["error"]:
        return _miss(name, "SQL 에서 오류가 났습니다. 붉은 글씨 중 'Error:' 가 들어간 줄을 읽고 고친 뒤 다시 실행하세요.", exp)
    if not isinstance(df, _pd.DataFrame):
        return _miss(name, "결과가 표가 아닙니다. SQL 이 SELECT 로 끝나는지 확인하세요.", exp)
    got = _sigs(df)
    if got["sig"] == exp["sig"] and (not exp.get("ordered") or got["osig"] == exp["osig"]):
        _RESULT[name] = True
        print(f"✅ {name} 정답!  (행 {got['rows']}개)"); return
    zero = "값의 철자 · 대소문자 · 따옴표(예: 'COMPLETED')와, 빈 값을 = 나 <> 로 비교하지 않았는지(IS NULL) 확인하세요."
    if got["sig"] == exp["sig"]:
        why = "값은 맞지만 행의 순서가 다릅니다. 정렬(ORDER BY · DESC)을 확인하세요."
    elif got["cols"] != exp["cols"]:
        if len(got["cols"]) != len(exp["cols"]):
            why = f"열 개수가 다릅니다 (기대 {len(exp['cols'])}개: {', '.join(exp['cols'])} / 현재 {len(got['cols'])}개)."
        elif sorted(got["cols"]) == sorted(exp["cols"]):
            why = f"열의 순서가 다릅니다. 문항에 적힌 순서대로 적으세요: {', '.join(exp['cols'])}"
        else:
            why = f"열 이름이 다릅니다. 기대: {', '.join(exp['cols'])}  /  현재: {', '.join(got['cols'])}  (AS 로 이름을 맞추세요)"
    elif got["rows"] == 0 and exp["rows"] > 0:
        why = "결과가 0행입니다. " + zero
    elif got["rows"] == 1 and len(df.columns) and all(_v(x) in ("0", "") for x in df.iloc[0]):
        why = "결과가 0 이거나 비어 있습니다. 조건에 맞는 행이 없다는 뜻입니다. " + zero
    elif got["rows"] != exp["rows"]:
        why = f"행 수가 다릅니다 (기대 {exp['rows']}행, 현재 {got['rows']}행). 조건과 LIMIT 을 확인하세요."
    elif exp.get("ordered"):
        why = "행 수는 맞지만 값이나 순서가 다릅니다. 조건과 정렬(ORDER BY · DESC)을 확인하세요."
    else:
        why = "행 수는 맞지만 값이 다릅니다. 조건이나 계산식을 확인하세요."
    _miss(name, why, exp)

def 진행확인():
    """지금까지 확인한 문항의 결과를 한눈에 봅니다. 선택 문항은 개수에서 뺍니다."""
    names = list(ANSWERS)
    core = [n for n in names if not ANSWERS[n].get("optional")]
    ok = [n for n in core if _RESULT.get(n) is True]
    print(f"정답 {len(ok)} / {len(core)} 문항")
    for n in names:
        mark = {True: "✅", False: "❌"}.get(_RESULT.get(n), "·  (아직 실행 안 함)")
        print(f"  {n:>4}  {mark}" + ("   (선택)" if ANSWERS[n].get("optional") else ""))

print("연결 완료:", DB)

ANSWERS = _load_answers("eNq1k99r01AUx/+VS162gZMm6S/3VnzxdRMEESmxTbVQVlkHPoxBKSlu68R2NmvUJHRil65U7cylRqz+QTkn/4P3pq2mRdEJPuXme84h5/P93uwJm6KwQfaESvEhewpxRVElMSklUg/SwjUilH+j75SfVJgusmOuXOLHe0LwlOLrpn9J8cAQ7rPKo+L2Lh+GxlvoeQTcFqujOYEzk6Dx3v/YJ/5II9M5tOvQOGCt15l4gpZG0G6hVYX2CA0atEw+5Y+aeHw46wv328mrO2qefaWglCoqVx7vFsvbSmku7TNtU/r/jLUuOiYctf+4NlmRYlJ8XZTWZXGFoEXhghI5htaEQ7MOAoMvzCxvboxFCWr2T3+uwC1HuVUplcsnC2pCUeUo97L+K24YazDo8yo7od1cYL9LODd88OB8SHxXC149C3QWsV1n668G+jvUWfnF5Vo041mbgU6VoH7ELCM4+Mot8N0unP8DbDwKK95QEkklUVByyWQUdlmfwcpRWP/TBIYUz0a8gd3QwPCwNhTR8qDxeTF1twXPX7LowKXYaaJmktlF6E1YlDSk6tAI93QCnGMCjo3jNg9/NuLwDhKcHvLEYVwNTqp4SqF2cXUrElErpEI6lo+J6ZycWrBiWZ/nLi0EH+4bBu/SoGOg2Q0Mnb8zS8Cxpnd0wZTw9zUIvqlDzwiNGLX9bx6/I4HeZz/ODzs2SDabXWX9JHOb3Mls3byV2Vr7W8797x1KENE=")   # 자동 확인용 (정답 SQL 은 없음)

## 데이터 구성 — 가상 병원 CDW

실제 병원 자료가 아니라 수업용으로 만든 가상 자료입니다 (2024년 · 병동 6곳). 열 이름이 생각나지 않으면 이 표로 돌아오세요.

| 표 | 한 행이 뜻하는 것 | 기본키 (PK) | 외래키 (FK) | 주로 쓰는 열 |
|---|---|---|---|---|
| `환자` | 환자 1명 | `환자ID` | — | 성별, 생년월일, 사망일 |
| `입원` | 입원 1건 | `입원ID` | `환자ID` → 환자, `병동` → 병동 | 병동, 진료과, 입원일, 퇴원일, 퇴원형태, 계획입원, 보험유형 |
| `진단` | 진단 1건 | `진단ID` | `입원ID` → 입원 | 진단코드(KCD), 주진단여부 |
| `검사` | 검사 1건 | `검사ID` | `환자ID` → 환자 | 검사일, 검사명, 결과값 |
| `병동` | 병동 1곳 | `병동` | — | 구분, 주진료과, 병상수 |
| `병동일보` | 병동 × 날짜 | `병동 + 날짜` | `병동` → 병동 | 재원환자수(자정 기준), 입원환자수, 퇴원환자수 |
| `근무` | 병동 × 날짜 × 근무조 | `병동 + 근무일 + 근무조` | `병동` → 병동 | 근무조(D · E · N), 간호사수 |
| `진료비` | 입원 × 항목 | `진료비ID` | `입원ID` → 입원 | 항목, 금액, 급여구분 |
| `낙상` | 낙상 1건 | `낙상ID` | `입원ID` → 입원 | 발생일, 근무조, 손상정도 |

- 환자 1명이 여러 번 입원하면 `입원` 표에 여러 행이 있습니다 (1 : N). 입원 1건에 진료비 · 진단 · 낙상도 여러 행일 수 있습니다.
- 표를 이을 때는 `ON 낙상.입원ID = 입원.입원ID` 처럼 외래키와 기본키를 맞춥니다.
- 2024-12-31 에 아직 재원 중인 입원은 `퇴원일` 이 비어 있습니다(NULL). 환자ID 는 앞자리 0 이 있는 글자입니다.

## 1. 지표 정의서 — 30일 재입원율

| 항목 | 내용 |
|---|---|
| 분모 | 2024년 퇴원 건 중 사망 · 전원 제외, 관찰 기간(30일)이 다 찬 퇴원 (12월 1일까지) |
| 분자 | 퇴원 다음 날부터 30일 안에 계획하지 않은 재입원(계획입원 = 'N')이 있는 퇴원 건 |
| 세는 단위 · 기준일 | 퇴원 건 · 퇴원일 |

## 2. 30일 재입원율 — 단계별로

In [ ]:
%%sql
-- [2번] 1단계 분모 — 사망 · 전원 제외 → 8,887건
SELECT COUNT(*) AS 퇴원건수
FROM 입원
WHERE 퇴원일 BETWEEN '2024-01-01' AND '2024-12-31'
  AND 퇴원형태 NOT IN ('사망', '전원');

In [ ]:
%%sql
-- [3번] 2단계 관찰 기간이 다 찬 퇴원만 → 8,137건 (12월 2일 이후 퇴원 750건 빠짐)
SELECT COUNT(*) AS 퇴원건수
FROM 입원
WHERE 퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'
  AND 퇴원형태 NOT IN ('사망', '전원');

In [ ]:
%%sql
-- [4번] 3단계 분자 — 입원 표를 두 번 (퇴원 · 다음입원) → 분모 8,137 · 분자 582 (7.2%)
SELECT COUNT(DISTINCT 퇴원.입원ID) AS 분모,
       COUNT(DISTINCT CASE WHEN 다음입원.입원ID IS NOT NULL
                 THEN 퇴원.입원ID END) AS 분자
FROM 입원 AS 퇴원
LEFT JOIN 입원 AS 다음입원
  ON 다음입원.환자ID = 퇴원.환자ID
  AND 다음입원.입원일 BETWEEN 퇴원.퇴원일 + 1
                       AND 퇴원.퇴원일 + 30
  AND 다음입원.계획입원 = 'N'
WHERE 퇴원.퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'
  AND 퇴원.퇴원형태 NOT IN ('사망', '전원');

In [ ]:
%%sql
-- [5번] 4단계 병동별 분모 · 분자
SELECT 퇴원.병동, COUNT(DISTINCT 퇴원.입원ID) AS 분모,        COUNT(DISTINCT CASE WHEN 다음입원.입원ID IS NOT NULL                  THEN 퇴원.입원ID END) AS 분자
FROM 입원 AS 퇴원
LEFT JOIN 입원 AS 다음입원   ON 다음입원.환자ID = 퇴원.환자ID   AND 다음입원.입원일 BETWEEN 퇴원.퇴원일 + 1                        AND 퇴원.퇴원일 + 30   AND 다음입원.계획입원 = 'N'
WHERE 퇴원.퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'   AND 퇴원.퇴원형태 NOT IN ('사망', '전원')
GROUP BY 퇴원.병동
ORDER BY 퇴원.병동;

In [ ]:
%%sql
-- [6번] 계획입원까지 넣으면 분자 1,574건 — 정의가 숫자를 바꿉니다
SELECT COUNT(DISTINCT 퇴원.입원ID) AS 분모,
       COUNT(DISTINCT CASE WHEN 다음입원.입원ID IS NOT NULL
                 THEN 퇴원.입원ID END) AS 분자
FROM 입원 AS 퇴원
LEFT JOIN 입원 AS 다음입원
  ON 다음입원.환자ID = 퇴원.환자ID
  AND 다음입원.입원일 BETWEEN 퇴원.퇴원일 + 1
                       AND 퇴원.퇴원일 + 30
WHERE 퇴원.퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'
  AND 퇴원.퇴원형태 NOT IN ('사망', '전원');

## 3. 간호 업무량 — 간호사 1인당 환자 수

근무 표(병동 × 날짜 × 근무조)와 병동일보(병동 × 날짜)를 **두 열로** 잇습니다.

In [ ]:
%%sql
-- [7번] 51병동 1월 1일 근무표 — 그날 재원 33명
SELECT 근무조, 간호사수
FROM 근무
WHERE 병동 = '51병동'
  AND 근무일 = '2024-01-01'
ORDER BY 근무조;

In [ ]:
%%sql
-- [8번] 근무조별 간호사 1인당 환자 수 — 밤(N)이 가장 많음
SELECT 근무조,
       ROUND(SUM(재원환자수) * 1.0 / SUM(간호사수), 1)
         AS 간호사1인당
FROM 근무
JOIN 병동일보
  ON 근무.병동 = 병동일보.병동
 AND 근무.근무일 = 병동일보.날짜
GROUP BY 근무조
ORDER BY 근무조;

In [ ]:
%%sql
-- [9번] 병동별 밤 근무
SELECT 근무.병동,
       ROUND(SUM(재원환자수) * 1.0 / SUM(간호사수), 1)
         AS 간호사1인당
FROM 근무
JOIN 병동일보
  ON 근무.병동 = 병동일보.병동
 AND 근무.근무일 = 병동일보.날짜
WHERE 근무조 = 'N'
GROUP BY 근무.병동
ORDER BY 근무.병동;

## 4. 작은 칸 가리기

5건 미만인 칸은 '5 미만'으로 가려서 내보냅니다. 숫자와 글자를 한 열에 쓰려고 숫자를 글자로 바꿉니다(CAST).

In [ ]:
%%sql
-- [10번] 병동 × 보험유형 사망 퇴원 — 12칸 중 3칸을 가림
SELECT 병동, 보험유형,
       CASE WHEN COUNT(*) < 5 THEN '5 미만'
            ELSE CAST(COUNT(*) AS VARCHAR)
       END AS 사망퇴원
FROM 입원
WHERE 퇴원형태 = '사망'
GROUP BY 병동, 보험유형
ORDER BY 병동, 보험유형;

## 직접 해보기 — 5문항

`___` 를 채워 실행하면 ✅ / ❌ 가 나옵니다. 열 이름이나 값이 필요하면 바로 위 조회 셀을 먼저 실행하세요.

**Q1.** **제외 조건** — 분모에서 사망과 함께 뺄 퇴원형태 **값**을 채우세요 (바로 위 셀에서 값 확인).

In [ ]:
%%sql
-- [11번] 퇴원형태에 어떤 값이 있는지 보기 (재원 중이면 NULL)
SELECT DISTINCT 퇴원형태
FROM 입원;

In [ ]:
%%sql
-- [12번] Q1
SELECT COUNT(*) AS 퇴원건수
FROM 입원
WHERE 퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'
  AND 퇴원형태 NOT IN ('사망', ___);

**Q2.** **관찰 기간** — 퇴원 뒤 30일을 다 볼 수 있는 마지막 퇴원일은 `'2024-12-01'` 입니다. 채우세요.

In [ ]:
%%sql
-- [13번] Q2
SELECT COUNT(*) AS 퇴원건수
FROM 입원
WHERE 퇴원일 BETWEEN '2024-01-01' AND ___
  AND 퇴원형태 NOT IN ('사망', '전원');

**Q3.** **비계획 재입원** — 분자에는 계획하지 않은 재입원만 셉니다. 계획입원 **값**을 채우세요 (바로 위 셀에서 값 확인).

In [ ]:
%%sql
-- [14번] 계획입원에 어떤 값이 있는지 보기
SELECT DISTINCT 계획입원
FROM 입원;

In [ ]:
%%sql
-- [15번] Q3
SELECT COUNT(DISTINCT 퇴원.입원ID) AS 분모,
       COUNT(DISTINCT CASE WHEN 다음입원.입원ID IS NOT NULL
                 THEN 퇴원.입원ID END) AS 분자
FROM 입원 AS 퇴원
LEFT JOIN 입원 AS 다음입원
  ON 다음입원.환자ID = 퇴원.환자ID
  AND 다음입원.입원일 BETWEEN 퇴원.퇴원일 + 1
                       AND 퇴원.퇴원일 + 30
  AND 다음입원.계획입원 = ___
WHERE 퇴원.퇴원일 BETWEEN '2024-01-01' AND '2024-12-01'
  AND 퇴원.퇴원형태 NOT IN ('사망', '전원');

**Q4.** **두 열로 JOIN** — 근무 표의 근무일과 맞출 병동일보의 **열 이름**을 채우세요 (바로 위 셀에서 열 이름 확인).

In [ ]:
%%sql
-- [16번] 병동일보의 열 이름 보기
SELECT *
FROM (DESCRIBE 병동일보);

In [ ]:
%%sql
-- [17번] Q4
SELECT 근무조,
       ROUND(SUM(재원환자수) * 1.0 / SUM(간호사수), 1)
         AS 간호사1인당
FROM 근무
JOIN 병동일보
  ON 근무.병동 = 병동일보.병동
 AND 근무.근무일 = 병동일보.___
GROUP BY 근무조
ORDER BY 근무조;

**Q5.** **작은 칸 가리기** — 숫자를 글자로 바꾸는 **함수 이름**(네 글자)을 채우세요.

In [ ]:
%%sql
-- [18번] Q5
SELECT 병동, 보험유형,
       CASE WHEN COUNT(*) < 5 THEN '5 미만'
            ELSE ___(COUNT(*) AS VARCHAR)
       END AS 사망퇴원
FROM 입원
WHERE 퇴원형태 = '사망'
GROUP BY 병동, 보험유형
ORDER BY 병동, 보험유형;

## 오늘 배운 것

- 지표는 지표 정의서(분모 · 분자 · 제외 · 기준일)부터 적는다
- 검산표로 단계마다 건수를 남겨 숫자를 설명한다
- 결과에는 기준 · 한계를 적고, 작은 칸은 가려서 내보낸다

다음 시간: 14강. 빅데이터와 의료데이터 표준 — OMOP CDM과 데이터 품질

In [ ]:
진행확인()   # [19번]